# LeetCode 2210 — Count Hills and Valleys in an Array bằng Python

Link đề: https://leetcode.com/problems/count-hills-and-valleys-in-an-array/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
class Solution:
    def countHillValley(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **đồi (hill)** và **thung lũng (valley)** khi các phần tử bằng nhau đứng liền kề tạo thành một "cao nguyên".
2. Biết cách xử lý **phần tử trùng liền kề**: nén chúng lại (dedupe) hoặc bỏ qua, để một cao nguyên chỉ bị đếm **một lần**.
3. Viết 4 cách: brute force quét hai phía, nén dãy rồi xét bộ ba, một lượt với biến `left`, và đếm **đổi dấu** của hiệu liên tiếp.
4. Dùng `itertools.groupby`, `zip`, hàm dấu `sign` quen thuộc.
5. Tự trace, tự test ngẫu nhiên và tránh các lỗi kinh điển: đếm trùng cao nguyên, xét nhầm hai đầu mảng, so sánh với hàng xóm trùng giá trị.

## 2. Đề bài & ví dụ

Cho mảng `nums` (đánh số từ 0). Một chỉ số `i` là **đồi** nếu:

- hàng xóm **khác giá trị gần nhất** bên trái nhỏ hơn `nums[i]`, và
- hàng xóm **khác giá trị gần nhất** bên phải nhỏ hơn `nums[i]`.

Tương tự, `i` là **thung lũng** nếu hai hàng xóm khác giá trị gần nhất (trái và phải) đều **lớn hơn** `nums[i]`.

Chỉ số **không có** hàng xóm khác giá trị ở một trong hai phía thì **không** là đồi hay thung lũng. Hai chỉ số liền kề có cùng giá trị thì được coi là **cùng một** đồi/thung lũng. Hãy trả về **số đồi + thung lũng**.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,4,1,1,6,5]` | `3` | `4` là đồi, `1,1` là một thung lũng, `6` là đồi. |
| 2 | `[6,6,5,5,4,1]` | `0` | Dãy chỉ giảm, không có đồi/thung lũng. |

Constraints:

- `3 <= nums.length <= 100`
- `1 <= nums[i] <= 100`

Điểm tinh tế: ở ví dụ 1, hai số `1,1` cùng tạo **một** thung lũng, không phải hai.

In [1]:
examples = [([2, 4, 1, 1, 6, 5], 3), ([6, 6, 5, 5, 4, 1], 0)]
for nums, expected in examples:
    print("nums =", nums, "| mong đợi =", expected)

nums = [2, 4, 1, 1, 6, 5] | mong đợi = 3
nums = [6, 6, 5, 5, 4, 1] | mong đợi = 0


## 3. Kiến thức nền

### 3.1. So sánh chuỗi và dấu

Python cho phép so sánh dây chuyền `a < b > c`, nghĩa là `a < b and b > c`. Hàm **dấu** của hiệu `b - a`: dương (tăng), âm (giảm), `0` (bằng).

In [2]:
a, b, c = 1, 4, 2
print(a < b > c)             # b là đỉnh: True
print(c > b < a)             # False
def sign(x):
    return (x > 0) - (x < 0)
print(sign(5), sign(-3), sign(0))

True
False
1 -1 0


### 3.2. `zip(nums, nums[1:])` — các cặp liền kề

Cặp `(nums[i], nums[i+1])` cho phép xét hiệu hai phần tử liền kề mà không phải lo chỉ số vượt biên.

In [3]:
nums = [2, 4, 1, 1, 6, 5]
diffs = [b - a for a, b in zip(nums, nums[1:])]
print(diffs)
print([sign(d) for d in diffs])

[2, -3, 0, 5, -1]
[1, -1, 0, 1, -1]


### 3.3. `itertools.groupby` — gom các phần tử bằng nhau liền kề

`groupby(nums)` gom các phần tử **giống nhau và liền kề** thành từng "run". Muốn chỉ lấy giá trị của mỗi run, dùng `[k for k, _ in groupby(nums)]`. Đây chính là phép **nén** (dedupe) hàng xóm trùng.

In [4]:
from itertools import groupby

nums = [2, 4, 1, 1, 6, 5, 5, 5]
print([k for k, _ in groupby(nums)])
print([(k, len(list(g))) for k, g in groupby(nums)])

[2, 4, 1, 6, 5]
[(2, 1), (4, 1), (1, 2), (6, 1), (5, 3)]


### 3.4. Vòng `while` để đi tới hàng xóm khác giá trị

Để tìm hàng xóm khác giá trị gần nhất bên trái của `i`, ta lùi `l = i - 1` cho tới khi `nums[l] != nums[i]` (hoặc `l < 0`).

In [5]:
nums = [2, 4, 1, 1, 6, 5]
i = 3
l = i - 1
while l >= 0 and nums[l] == nums[i]:
    l -= 1
print("hàng xóm khác giá trị bên trái của i=3 là chỉ số", l, "giá trị", nums[l])

hàng xóm khác giá trị bên trái của i=3 là chỉ số 1 giá trị 4


## 4. Ý tưởng nền tảng — tính bằng tay

Lấy `nums = [2,4,1,1,6,5]`. Gạch bỏ phần tử trùng liền kề: `[2,4,1,6,5]`.

Giờ mỗi phần tử ở **giữa** (không phải đầu/cuối) có hai hàng xóm khác giá trị rõ ràng:

| Phần tử | Hàng xóm trái | Hàng xóm phải | Loại |
|---|---|---|---|
| `4` | `2` | `1` | cả hai nhỏ hơn → **đồi** |
| `1` | `4` | `6` | cả hai lớn hơn → **thung lũng** |
| `6` | `1` | `5` | cả hai nhỏ hơn → **đồi** |

Tổng `3`. Hai đầu (`2` và `5`) thiếu một phía nên không tính.

**Nhận xét quan trọng:** sau khi nén, bài biến thành "đếm các phần tử là đỉnh cục bộ hoặc đáy cục bộ trong dãy không có hai số liền kề bằng nhau".

In [6]:
compressed = [2, 4, 1, 6, 5]
count = 0
for i in range(1, len(compressed) - 1):
    l, m, r = compressed[i - 1], compressed[i], compressed[i + 1]
    if (m > l and m > r) or (m < l and m < r):
        count += 1
        print("tìm thấy", m, "giữa", l, "và", r)
print("tổng =", count)

tìm thấy 4 giữa 2 và 1
tìm thấy 1 giữa 4 và 6
tìm thấy 6 giữa 1 và 5
tổng = 3


## 5. Cách 1 — Brute Force: quét hai phía cho mỗi chỉ số

Theo đúng đề: với mỗi `i`, đi sang trái tìm hàng xóm khác giá trị gần nhất, đi sang phải làm tương tự, rồi so sánh. Để một cao nguyên không bị đếm nhiều lần, ta **chỉ xét `i` đầu tiên** của mỗi cao nguyên (tức `i == 0` hoặc `nums[i] != nums[i-1]`).

In [7]:
class SolutionBrute:
    def countHillValley(self, nums):
        n, count = len(nums), 0
        for i in range(n):
            if i > 0 and nums[i] == nums[i - 1]:
                continue                       # chỉ xét phần tử đầu của mỗi cao nguyên
            l = i - 1
            while l >= 0 and nums[l] == nums[i]:
                l -= 1
            r = i + 1
            while r < n and nums[r] == nums[i]:
                r += 1
            if l < 0 or r >= n:
                continue                       # thiếu hàng xóm khác giá trị
            if (nums[i] > nums[l] and nums[i] > nums[r]) or (nums[i] < nums[l] and nums[i] < nums[r]):
                count += 1
        return count

sol = SolutionBrute()
print(sol.countHillValley([2, 4, 1, 1, 6, 5]))
print(sol.countHillValley([6, 6, 5, 5, 4, 1]))

3
0


**Độ phức tạp:** mỗi `i` có thể quét tới `O(n)` (khi mảng là một cao nguyên dài) ⇒ Time `O(n²)` tệ nhất, Space `O(1)`.

**Nút thắt:** ta quét lại các phần tử trùng nhiều lần. Chỉ cần một lượt nếu **nhớ giá trị hàng xóm khác giá trị bên trái** thay vì tìm lại.

## 6. Ý tưởng thuật toán — bước đột phá

**Insight:** đồi/thung lũng chỉ phụ thuộc vào **chiều** thay đổi: đang đi lên rồi chuyển sang đi xuống (đồi) hoặc ngược lại (thung lũng). Phần tử bằng nhau **không** làm đổi chiều, nên bỏ qua.

Có hai cách rút gọn:

1. **Nén** các phần tử trùng liền kề, rồi chỉ kiểm tra bộ ba `(l, m, r)`.
2. **Duy trì `left`** = giá trị khác giá trị hiện tại gần nhất bên trái; khi `nums[i] != nums[i+1]` thì `nums[i+1]` chính là hàng xóm phải khác giá trị.

Quy tắc "nếu… thì…":

- **Nếu** `nums[i] != nums[i+1]` (hết cao nguyên) **và** `nums[i]` cùng lớn hơn hoặc cùng nhỏ hơn `left` và `nums[i+1]` **thì** đếm thêm `1`, rồi `left = nums[i]`.
- **Nếu** `nums[i] == nums[i+1]` **thì** bỏ qua, **không** cập nhật `left`.

**Invariant:** tại bước `i`, `left` luôn là giá trị khác `nums[i]` gần nhất bên trái (hoặc `nums[0]` khi chưa có).

## 7. Các cách tiếp theo

### Cách 2 — Nén dãy bằng `groupby`, rồi xét bộ ba

Sau khi nén, mọi phần tử giữa đều có hàng xóm khác giá trị hai phía. Chỉ việc kiểm tra đỉnh/đáy.

In [8]:
from itertools import groupby

class SolutionCompress:
    def countHillValley(self, nums):
        arr = [k for k, _ in groupby(nums)]        # bỏ trùng liền kề
        count = 0
        for i in range(1, len(arr) - 1):
            if (arr[i] > arr[i - 1] and arr[i] > arr[i + 1]) or \
               (arr[i] < arr[i - 1] and arr[i] < arr[i + 1]):
                count += 1
        return count

sol = SolutionCompress()
print(sol.countHillValley([2, 4, 1, 1, 6, 5]))
print(sol.countHillValley([6, 6, 5, 5, 4, 1]))

3
0


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho dãy nén.

### Cách 3 — Một lượt với biến `left` (không tạo mảng phụ)

Giữ `left` là giá trị khác gần nhất bên trái. Chỉ khi `nums[i] != nums[i+1]` ta mới "chốt" một cao nguyên và so sánh.

In [9]:
class SolutionOnePass:
    def countHillValley(self, nums):
        count = 0
        left = nums[0]
        for i in range(1, len(nums) - 1):
            if nums[i] != nums[i + 1]:            # nums[i+1] là hàng xóm phải khác giá trị
                if (nums[i] > left and nums[i] > nums[i + 1]) or \
                   (nums[i] < left and nums[i] < nums[i + 1]):
                    count += 1
                left = nums[i]
        return count

sol = SolutionOnePass()
print(sol.countHillValley([2, 4, 1, 1, 6, 5]))
print(sol.countHillValley([6, 6, 5, 5, 4, 1]))

3
0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Lưu ý: khi `nums[i] == left` (đầu mảng toàn số bằng nhau) cả hai điều kiện `>` và `<` đều sai ⇒ không đếm nhầm.

### Cách 4 — Đếm số lần **đổi dấu** của hiệu

Xét dãy hiệu `d[i] = nums[i+1] - nums[i]`, **bỏ các số 0**. Mỗi lần dấu đổi (từ `+` sang `-` hoặc ngược lại) là đúng một đồi hoặc một thung lũng.

Ví dụ `[2,4,1,1,6,5]`: hiệu `+2, -3, 0, +5, -1` → bỏ `0`: `+, -, +, -` → đổi dấu `3` lần.

In [10]:
class SolutionSign:
    def countHillValley(self, nums):
        prev_sign, count = 0, 0
        for a, b in zip(nums, nums[1:]):
            s = (b > a) - (b < a)                  # +1, -1 hoặc 0
            if s == 0:
                continue                           # bằng nhau: không đổi chiều
            if prev_sign != 0 and s != prev_sign:
                count += 1
            prev_sign = s
        return count

sol = SolutionSign()
print(sol.countHillValley([2, 4, 1, 1, 6, 5]))
print(sol.countHillValley([6, 6, 5, 5, 4, 1]))

3
0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Cách này cho thấy bản chất: "số đồi + thung lũng = số lần đổi chiều". Hai đầu mảng tự động bị loại vì chưa có `prev_sign` (trước nó chưa có chiều nào).

## 8. TRACE từng bước

Trace cách 3 với `nums = [2,4,1,1,6,5]`:

| i | nums[i] | nums[i+1] | left | nums[i] != nums[i+1]? | đồi/thung lũng? | count | left mới |
|---|---|---|---|---|---|---|---|
| 1 | 4 | 1 | 2 | có | `4>2` và `4>1` → đồi | 1 | 4 |
| 2 | 1 | 1 | 4 | không | bỏ qua | 1 | 4 |
| 3 | 1 | 6 | 4 | có | `1<4` và `1<6` → thung lũng | 2 | 1 |
| 4 | 6 | 5 | 1 | có | `6>1` và `6>5` → đồi | 3 | 6 |

Hàm trace in đúng bảng này.

In [11]:
def trace_one_pass(nums):
    count = 0
    left = nums[0]
    for i in range(1, len(nums) - 1):
        cur, nxt = nums[i], nums[i + 1]
        if cur != nxt:
            kind = "-"
            if cur > left and cur > nxt:
                kind = "đồi"; count += 1
            elif cur < left and cur < nxt:
                kind = "thung lũng"; count += 1
            print(f"i={i} cur={cur} left={left} nxt={nxt} -> {kind} | count={count}")
            left = cur
        else:
            print(f"i={i} cur={cur} == nxt={nxt} -> bỏ qua | count={count} left={left}")
    return count

print(trace_one_pass([2, 4, 1, 1, 6, 5]))

i=1 cur=4 left=2 nxt=1 -> đồi | count=1
i=2 cur=1 == nxt=1 -> bỏ qua | count=1 left=4
i=3 cur=1 left=4 nxt=6 -> thung lũng | count=2
i=4 cur=6 left=1 nxt=5 -> đồi | count=3
3


In [12]:
# cao nguyên ở đầu và cuối: không có đồi/thung lũng giả
print(trace_one_pass([3, 3, 5, 5, 2, 2]))

i=1 cur=3 left=3 nxt=5 -> - | count=0
i=2 cur=5 == nxt=5 -> bỏ qua | count=0 left=3
i=3 cur=5 left=3 nxt=2 -> đồi | count=1
i=4 cur=2 == nxt=2 -> bỏ qua | count=1 left=5
1


## 9. Code nộp LeetCode

Dùng cách 3 (một lượt, `O(1)` bộ nhớ).

In [13]:
from typing import List

class Solution:
    def countHillValley(self, nums: List[int]) -> int:
        count = 0
        left = nums[0]
        for i in range(1, len(nums) - 1):
            if nums[i] != nums[i + 1]:
                if (nums[i] > left and nums[i] > nums[i + 1]) or \
                   (nums[i] < left and nums[i] < nums[i + 1]):
                    count += 1
                left = nums[i]
        return count

print(Solution().countHillValley([2, 4, 1, 1, 6, 5]))
print(Solution().countHillValley([6, 6, 5, 5, 4, 1]))

3
0


## 10. Vì sao thuật toán đúng?

- **Mỗi cao nguyên đếm đúng một lần:** ta chỉ kiểm tra tại phần tử **cuối** của cao nguyên (nơi `nums[i] != nums[i+1]`), nên các bản sao bằng nhau không bị đếm lặp.
- **Hàng xóm phải đúng:** tại phần tử cuối cao nguyên, `nums[i+1]` khác `nums[i]` và đứng sát ⇒ là hàng xóm khác giá trị gần nhất bên phải.
- **Hàng xóm trái đúng:** `left` chỉ được cập nhật tại phần tử cuối của một cao nguyên (`left = nums[i]`), vậy khi tới cao nguyên kế tiếp, `left` là giá trị của cao nguyên trước đó, tức hàng xóm khác giá trị gần nhất bên trái.
- **Hai đầu bị loại:** vòng lặp bắt đầu từ `i = 1` và dừng ở `n - 2`; một cao nguyên chạm biên không có hàng xóm khác giá trị phía biên nên không hợp lệ. Cao nguyên đầu có `left = nums[0]` bằng chính nó ⇒ không thỏa `>` hay `<`. Cao nguyên cuối không bao giờ có `nums[i] != nums[i+1]` khi `i < n-1` bên trong cao nguyên ⇒ không bị đếm.
- **Cách đổi dấu** tương đương: đồi/thung lũng đúng là chỗ chiều biến thiên đảo ngược.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` tệ nhất (cao nguyên dài), thường `O(n)` | `O(1)` |
| 2. Nén `groupby` | `O(n)` | `O(n)` |
| 3. Một lượt `left` | `O(n)` | `O(1)` |
| 4. Đổi dấu | `O(n)` | `O(1)` |

Với `n <= 100` mọi cách đều nhanh; điều quan trọng là **tư duy xử lý cao nguyên**.

## 12. Bộ test

Test ví dụ đề, edge case, rồi đối chiếu 5 lớp giải trên input ngẫu nhiên có `seed`.

In [14]:
solutions = [SolutionBrute(), SolutionCompress(), SolutionOnePass(), SolutionSign(), Solution()]

for s in solutions:
    assert s.countHillValley([2, 4, 1, 1, 6, 5]) == 3
    assert s.countHillValley([6, 6, 5, 5, 4, 1]) == 0
print("ví dụ đề: OK")

ví dụ đề: OK


In [15]:
for s in solutions:
    assert s.countHillValley([1, 2, 3]) == 0                    # chỉ tăng
    assert s.countHillValley([1, 3, 2]) == 1                    # một đồi
    assert s.countHillValley([3, 1, 2]) == 1                    # một thung lũng
    assert s.countHillValley([5, 5, 5, 5]) == 0                 # toàn bằng nhau
    assert s.countHillValley([1, 3, 3, 3, 2]) == 1              # cao nguyên là một đồi
    assert s.countHillValley([4, 4, 1, 1, 4, 4]) == 1           # thung lũng ở giữa hai cao nguyên
    assert s.countHillValley([1, 5, 1, 5, 1, 5, 1]) == 5        # răng cưa
    assert s.countHillValley([2, 2, 3, 3]) == 0                 # cao nguyên ở hai đầu
print("edge case: OK")

edge case: OK


In [16]:
import random
random.seed(2210)
count = 0
for _ in range(1000):
    n = random.randint(3, 20)
    nums = [random.randint(1, 4) for _ in range(n)]     # miền nhỏ => nhiều giá trị trùng
    results = [s.countHillValley(nums) for s in solutions]
    assert len(set(results)) == 1, (nums, results)
    count += 1
print("random đối chiếu:", count, "ca x", len(solutions), "cách: OK")

random đối chiếu: 1000 ca x 5 cách: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ so với hàng xóm sát vách (bỏ qua giá trị trùng)

Viết `nums[i] > nums[i-1] and nums[i] > nums[i+1]` là sai khi đồi/thung lũng có **nhiều** phần tử bằng nhau — không phần tử nào thỏa.

In [17]:
def wrong_adjacent(nums):
    count = 0
    for i in range(1, len(nums) - 1):
        if (nums[i] > nums[i - 1] and nums[i] > nums[i + 1]) or \
           (nums[i] < nums[i - 1] and nums[i] < nums[i + 1]):
            count += 1
    return count

nums = [2, 4, 1, 1, 6, 5]
print("sai :", wrong_adjacent(nums))        # bỏ lỡ thung lũng "1,1"
print("đúng:", Solution().countHillValley(nums))

sai : 2
đúng: 3


### Lỗi 2 — Đếm trùng cao nguyên

Nếu "sửa" bằng cách tìm hàng xóm khác giá trị cho **mọi** `i` mà không chỉ xét một phần tử mỗi cao nguyên, cao nguyên dài bị đếm nhiều lần.

In [18]:
def wrong_double_count(nums):
    n, count = len(nums), 0
    for i in range(n):                       # thiếu: if i > 0 and nums[i] == nums[i-1]: continue
        l = i - 1
        while l >= 0 and nums[l] == nums[i]:
            l -= 1
        r = i + 1
        while r < n and nums[r] == nums[i]:
            r += 1
        if l < 0 or r >= n:
            continue
        if (nums[i] > nums[l] and nums[i] > nums[r]) or (nums[i] < nums[l] and nums[i] < nums[r]):
            count += 1
    return count

nums = [1, 3, 3, 3, 1]
print("sai :", wrong_double_count(nums))     # 3 (mỗi số 3 đều đếm)
print("đúng:", Solution().countHillValley(nums))

sai : 3
đúng: 1


### Lỗi 3 — Cập nhật `left` ở cả bước bỏ qua

Nếu viết `left = nums[i]` ngoài khối `if nums[i] != nums[i+1]`, thì ở giữa cao nguyên `left` bị gán bằng chính giá trị cao nguyên và so sánh sau đó sai.

In [19]:
def wrong_left_update(nums):
    count, left = 0, nums[0]
    for i in range(1, len(nums) - 1):
        if nums[i] != nums[i + 1]:
            if (nums[i] > left and nums[i] > nums[i + 1]) or (nums[i] < left and nums[i] < nums[i + 1]):
                count += 1
        left = nums[i]                       # SAI: cập nhật cả khi nums[i] == nums[i+1]
    return count

nums = [1, 5, 5, 2]
print("sai :", wrong_left_update(nums))      # 0: left bị ghi đè thành 5
print("đúng:", Solution().countHillValley(nums))   # 1

sai : 0
đúng: 1


### Lỗi 4 — Đếm cả hai đầu mảng

Phần tử đầu/cuối thiếu một phía nên không hợp lệ. Nếu vòng lặp chạy `range(n)` và tự gán hàng xóm ngoài biên bằng `0`/`-inf`, kết quả bị thừa.

In [20]:
def wrong_edges(nums):
    padded = [0] + nums + [0]                 # SAI: coi ngoài biên là 0
    count = 0
    for i in range(1, len(padded) - 1):
        if padded[i] > padded[i - 1] and padded[i] > padded[i + 1]:
            count += 1
    return count

nums = [1, 2, 3]
print("sai :", wrong_edges(nums))             # 1 (đầu/cuối bị coi là đồi)
print("đúng:", Solution().countHillValley(nums))   # 0

sai : 1
đúng: 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hàng xóm **khác giá trị** gần nhất" ⇒ nén trùng liền kề hoặc nhớ biến `left`.
- Đồi/thung lũng = **đổi chiều**; đếm đổi chiều bằng hiệu liên tiếp bỏ số `0`.
- Luôn nghĩ tới trường hợp **toàn bằng nhau**, **cao nguyên ở hai đầu**.
- Bài họ hàng: Monotonic Array, Valid Mountain Array, Peak Index in a Mountain Array, Find Peak Element, Wiggle Subsequence.
- Công thức nhớ nhanh: `groupby` để nén; `(b > a) - (b < a)` để lấy dấu.

## 15. Biến thể của bài

1. **Chỉ đếm đồi** (hoặc chỉ thung lũng): tách điều kiện.
2. **Trả về vị trí** của đồi/thung lũng (chỉ số đầu của cao nguyên).
3. **Độ dốc tối thiểu:** chỉ đếm khi `|nums[i] - left| >= d` và `|nums[i] - nums[i+1]| >= d`.
4. **Mảng vòng:** phần tử đầu và cuối cũng là hàng xóm của nhau.
5. **Độ dài dãy đồi-thung lũng xen kẽ dài nhất** (liên quan Wiggle Subsequence, LC 376).

In [21]:
# Biến thể 1: đếm riêng số đồi và số thung lũng
def count_hills_and_valleys(nums):
    hills = valleys = 0
    left = nums[0]
    for i in range(1, len(nums) - 1):
        if nums[i] != nums[i + 1]:
            if nums[i] > left and nums[i] > nums[i + 1]:
                hills += 1
            elif nums[i] < left and nums[i] < nums[i + 1]:
                valleys += 1
            left = nums[i]
    return hills, valleys

print(count_hills_and_valleys([2, 4, 1, 1, 6, 5]))   # (2, 1)

(2, 1)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Quét trái/phải tìm hàng xóm khác giá trị | `O(n²)` tệ nhất | `O(1)` | Hiểu đúng định nghĩa đề |
| 2. Nén `groupby` | Bỏ trùng liền kề rồi xét bộ ba | `O(n)` | `O(n)` | Dễ đọc, dễ chứng minh |
| 3. Một lượt `left` | Nhớ hàng xóm trái khác giá trị | `O(n)` | `O(1)` | **Nộp bài**, tối ưu |
| 4. Đổi dấu | Đếm số lần đổi chiều của hiệu | `O(n)` | `O(1)` | Nhìn ra bản chất, mở rộng cho bài khác |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countHillValley([1,3,2,2,5,5,4])` bằng bao nhiêu?

**Bài 2 (tính tay).** Nén `[7,7,2,2,2,9,9,3,3,3,3,8]` rồi đếm đồi/thung lũng.

**Bài 3 (điền TODO).** Hoàn thành `compress(nums)` trả về dãy sau khi bỏ trùng liền kề (không dùng `groupby`).

**Bài 4 (tự viết lại).** Viết `count_sign_changes(nums)` đếm số lần đổi dấu của hiệu (bỏ qua số `0`) mà không nhìn cách 4.

**Bài 5 (mini challenge).** Viết `hill_valley_positions(nums)` trả về danh sách chỉ số **đầu tiên** của mỗi cao nguyên là đồi hoặc thung lũng.

In [22]:
# Bài 1: đoán rồi chạy kiểm tra
print(Solution().countHillValley([1, 3, 2, 2, 5, 5, 4]))

3


In [23]:
# Bài 3, 4, 5: TODO - KHÔNG được gọi khi run all
def compress(nums):
    # TODO: bỏ trùng liền kề, giữ một đại diện cho mỗi cao nguyên
    pass

def count_sign_changes(nums):
    # TODO: đếm số lần đổi dấu của hiệu liên tiếp, bỏ qua hiệu bằng 0
    pass

def hill_valley_positions(nums):
    # TODO: trả về chỉ số đầu của mỗi cao nguyên là đồi/thung lũng
    pass

## 18. Gợi ý 3 mức

**Bài 3**
- Mức 1: Bắt đầu với `res = []`.
- Mức 2: Thêm `x` nếu `res` rỗng hoặc `x != res[-1]`.
- Mức 3: `for x in nums: if not res or x != res[-1]: res.append(x)`.

**Bài 4**
- Mức 1: Dùng biến `prev` lưu dấu khác `0` gần nhất.
- Mức 2: Bỏ qua cặp có `a == b`.
- Mức 3: Khi dấu mới khác `prev` và `prev != 0` thì cộng `1`, rồi gán `prev = dấu mới`.

**Bài 5**
- Mức 1: Giống cách 3 nhưng cần nhớ chỉ số đầu cao nguyên `start`.
- Mức 2: Khi bắt đầu một cao nguyên mới (`nums[i] != nums[i-1]`), ghi `start = i`.
- Mức 3: Khi chốt cao nguyên là đồi/thung lũng, `res.append(start)`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [24]:
# Bài 1: nén [1,3,2,5,4] -> 3 (đồi), 2 (thung lũng), 5 (đồi) = 3
assert Solution().countHillValley([1, 3, 2, 2, 5, 5, 4]) == 3

# Bài 2: nén [7,2,9,3,8] -> 2 thung lũng, 9 đồi, 3 thung lũng = 3
def compress_ans(nums):
    res = []
    for x in nums:
        if not res or x != res[-1]:
            res.append(x)
    return res
assert compress_ans([7, 7, 2, 2, 2, 9, 9, 3, 3, 3, 3, 8]) == [7, 2, 9, 3, 8]
assert Solution().countHillValley([7, 7, 2, 2, 2, 9, 9, 3, 3, 3, 3, 8]) == 3

# Bài 3
assert compress_ans([]) == []
assert compress_ans([1, 1, 1]) == [1]

# Bài 4
def count_sign_changes_ans(nums):
    prev, cnt = 0, 0
    for a, b in zip(nums, nums[1:]):
        s = (b > a) - (b < a)
        if s == 0:
            continue
        if prev != 0 and s != prev:
            cnt += 1
        prev = s
    return cnt
assert count_sign_changes_ans([2, 4, 1, 1, 6, 5]) == 3
assert count_sign_changes_ans([6, 6, 5, 5, 4, 1]) == 0

# Bài 5
def hill_valley_positions_ans(nums):
    res, left, start = [], nums[0], 0
    for i in range(1, len(nums) - 1):
        if nums[i] != nums[i - 1]:
            start = i
        if nums[i] != nums[i + 1]:
            if (nums[i] > left and nums[i] > nums[i + 1]) or (nums[i] < left and nums[i] < nums[i + 1]):
                res.append(start)
            left = nums[i]
    return res
assert hill_valley_positions_ans([2, 4, 1, 1, 6, 5]) == [1, 2, 4]
assert hill_valley_positions_ans([1, 2, 3]) == []
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
from typing import List

class Solution:
    def countHillValley(self, nums: List[int]) -> int:
        count = 0
        left = nums[0]                         # hàng xóm khác giá trị bên trái
        for i in range(1, len(nums) - 1):
            if nums[i] != nums[i + 1]:         # chốt cuối cao nguyên
                if (nums[i] > left and nums[i] > nums[i + 1]) or \
                   (nums[i] < left and nums[i] < nums[i + 1]):
                    count += 1
                left = nums[i]
        return count
```

### Checklist

- [ ] Hiểu cao nguyên (các số bằng nhau liền kề) chỉ tính **một** đồi/thung lũng.
- [ ] Hàng xóm cần so là hàng xóm **khác giá trị** gần nhất, không phải sát vách.
- [ ] `left` chỉ cập nhật khi `nums[i] != nums[i+1]`.
- [ ] Hai đầu mảng không bao giờ được đếm.
- [ ] Đã test toàn số bằng nhau, cao nguyên ở đầu/cuối, răng cưa.
- [ ] Biết cách nhìn "đếm số lần đổi chiều" làm cách kiểm chứng chéo.